# Getting Started with Engram

This notebook walks through the basics of Engram — ingesting conversations, searching memories, and understanding how the retrieval pipeline works.

## Install
```bash
pip install engram-search
```

In [ ]:
# Install if needed
# !pip install engram-search

## 1. Create a Memory Store

Engram uses FAISS + SQLite for local storage. No API keys, no cloud setup.

In [ ]:
from engram.backends.faiss_backend import FaissBackend
from engram.backends.base import Document
from engram.retrieval.embedder import Embedder
from engram.ingestion.parser import session_to_documents

# Initialize
backend = FaissBackend(path="./demo_store", dimension=1024)
embedder = Embedder("bge-large")

print(f"Store initialized. Documents: {backend.count()}")
print(f"Embedding model: bge-large-en-v1.5 (1024 dimensions)")

## 2. Ingest Conversations

Conversations are lists of `{role, content}` turns — the same format used by OpenAI, Anthropic, etc.

In [ ]:
# Sample conversations
conversations = [
    {
        "id": "chat_001",
        "timestamp": "2024-03-10",
        "turns": [
            {"role": "user", "content": "I've been learning Python for about 3 months now. I really enjoy working with pandas for data analysis."},
            {"role": "assistant", "content": "That's great progress! Pandas is an excellent library. Have you tried matplotlib or seaborn for visualization?"},
            {"role": "user", "content": "Yes! I made some scatter plots last week to analyze our sales data at work."},
            {"role": "assistant", "content": "Nice! Scatter plots are perfect for finding correlations in sales data."}
        ]
    },
    {
        "id": "chat_002",
        "timestamp": "2024-03-15",
        "turns": [
            {"role": "user", "content": "My dog Max has been acting weird lately. He's not eating his food."},
            {"role": "assistant", "content": "I'm sorry to hear that. Has Max had any changes in his routine or environment recently?"},
            {"role": "user", "content": "We moved to a new apartment last week. Maybe that's stressing him out."},
            {"role": "assistant", "content": "That's very likely the cause. Dogs can be sensitive to changes in environment. Give him a few days to adjust."}
        ]
    },
    {
        "id": "chat_003",
        "timestamp": "2024-03-20",
        "turns": [
            {"role": "user", "content": "I'm thinking about switching from JavaScript to TypeScript for our company's frontend."},
            {"role": "assistant", "content": "TypeScript is a great choice. The type system catches many bugs at compile time."},
            {"role": "user", "content": "Our team has 5 developers and we're using React with Next.js."},
            {"role": "assistant", "content": "Next.js has excellent TypeScript support built in. Migration can be done incrementally."}
        ]
    },
    {
        "id": "chat_004",
        "timestamp": "2024-03-25",
        "turns": [
            {"role": "user", "content": "I want to start running but I have bad knees. Any suggestions?"},
            {"role": "assistant", "content": "Consider starting with low-impact alternatives like swimming or cycling to build endurance."},
            {"role": "user", "content": "I used to swim in college actually. Maybe I should get back into it."},
            {"role": "assistant", "content": "That's a great idea! Swimming is one of the best full-body workouts and very easy on the joints."}
        ]
    }
]

print(f"Prepared {len(conversations)} conversations for ingestion")

In [ ]:
# Parse conversations into documents
all_docs = []
for conv in conversations:
    parsed = session_to_documents(
        session=conv["turns"],
        session_id=conv["id"],
        timestamp=conv["timestamp"],
        include_assistant=True,
    )
    all_docs.extend(parsed)
    print(f"  {conv['id']}: {len(parsed)} documents (session + synthetic)")

print(f"\nTotal documents: {len(all_docs)}")

In [ ]:
# Embed and store
texts = [d["text"] for d in all_docs]
embeddings = embedder.encode_documents(texts)

documents = [
    Document(
        id=doc["id"],
        text=doc["text"],
        embedding=embeddings[i].tolist(),
        metadata=doc["metadata"],
    )
    for i, doc in enumerate(all_docs)
]

backend.add(documents)
print(f"Ingested {len(documents)} documents. Store total: {backend.count()}")

## 3. Search Memories

Now let's query the store with natural language questions.

In [ ]:
def search(query, top_k=3):
    """Simple search helper."""
    query_vec = embedder.encode_query(query)
    results = backend.query(embedding=query_vec.tolist(), top_k=top_k)
    
    print(f'Query: "{query}"\n{"─" * 60}')
    for i, doc in enumerate(results, 1):
        meta = doc.metadata or {}
        print(f"  [{i}] Score: {doc.score:.3f} | Session: {meta.get('session_id', '?')} | {meta.get('timestamp', '')}")
        print(f"      {doc.text[:120]}...")
        print()
    return results

In [ ]:
# Test various queries
search("What programming languages does the user know?")

In [ ]:
search("Tell me about the user's pet")

In [ ]:
search("What exercise does the user do?")

In [ ]:
search("What tech stack is the team using?")

## 4. Hybrid Search with BM25

Dense embeddings are great for semantic similarity, but BM25 keyword matching catches exact terms that embeddings might miss. Engram fuses both via Reciprocal Rank Fusion (RRF).

In [ ]:
from engram.retrieval.sparse import BM25
from engram.retrieval.pipeline import reciprocal_rank_fusion
import numpy as np

query = "Max the dog"

# Dense ranking
query_vec = embedder.encode_query(query)
dense_results = backend.query(embedding=query_vec.tolist(), top_k=10)
dense_ranking = [(d.id, d.score) for d in dense_results]

# BM25 ranking
bm25 = BM25()
all_texts = [d.text for d in documents]
all_ids = [d.id for d in documents]
bm25_scores = bm25.score_query_against_docs(query, all_texts)
sparse_ranking = sorted(
    zip(all_ids, bm25_scores), key=lambda x: x[1], reverse=True
)

# RRF fusion
fused = reciprocal_rank_fusion(dense_ranking, sparse_ranking)

print(f'Query: "{query}"\n')
print("Top 5 after RRF fusion:")
id_to_doc = {d.id: d for d in documents}
for doc_id, score in fused[:5]:
    doc = id_to_doc[doc_id]
    meta = doc.metadata or {}
    print(f"  {score:.4f} | {meta.get('session_id', doc_id)} | {doc.text[:80]}...")

## 5. Understanding Synthetic Documents

Engram generates extra "synthetic" documents to improve recall:
- **Preference docs**: Extract user preferences and habits
- **Topic docs**: Key nouns for vocabulary bridging

These help match queries phrased differently from the original conversation.

In [ ]:
from engram.ingestion.parser import extract_preferences, extract_topics

sample_turns = conversations[0]["turns"]

prefs = extract_preferences(sample_turns)
topics = extract_topics(sample_turns)

print("Extracted preferences:")
for p in prefs:
    print(f"  - {p}")

print(f"\nExtracted topics: {topics}")

## Cleanup

In [ ]:
import shutil
shutil.rmtree("./demo_store", ignore_errors=True)
print("Demo store cleaned up.")